In [1]:
# !pip install -e /home/jovyan/PROJECTS/scale-aware-air-sea

In [2]:
#TODO: I want attrs to propagate through...need to start in the first step though...

In [3]:
import gcsfs
import xarray as xr
import numpy as np
from scale_aware_air_sea.utils import smooth_inputs_dataset, to_zarr_split, open_zarr
from scale_aware_air_sea.parameters import get_params

In [4]:
fs = gcsfs.GCSFileSystem()

# load global parameters
params = get_params('v0.6.2', test=True)  

mapper_smooth_fluxes = fs.get_mapper(params['paths']['CM26']['smooth_fluxes'])
mapper_coarse_fluxes = fs.get_mapper(params['paths']['CM26']['coarse_fluxes'])

# 👇 change this to not overwrite my work please
# mapper_smooth_flux_decomposition = fs.get_mapper(params['paths']['CM26']['smooth_decomposition'])
# mapper_coarse_flux_decomposition = fs.get_mapper(params['paths']['CM26']['coarse_decomposition'])

In [5]:
# new scale separation
def decomposition_smooth(ds):
    """Decomposing filtered flux outputs into different terms to evaluate"""
    def filt(ds):
        return smooth_inputs_dataset(ds, ['yt_ocean', 'xt_ocean'], 50)
    decomp = {}
    # Q_H (AB) - high resolution input
    decomp['Q_H'] = ds.sel(smoothing='smooth_none')
    decomp['Q_H_bar'] = filt(decomp['Q_H'])
    # Q_L low resolution input
    decomp['Q_L'] = ds.sel(smoothing='smooth_all')
    decomp['Q_L_bar'] = filt(ds.sel(smoothing='smooth_all'))
    decomp['Q_L_prime'] = decomp['Q_L'] - decomp['Q_L_bar'] # TODO: I could potentially compute this on the fly...
    
    # mixed low resolution input
    decomp['Q_L_ocean'] = ds.sel(smoothing='smooth_vel_tracer_ocean')
    decomp['Q_L_ocean_bar'] = filt(decomp['Q_L_ocean'])
    
    decomp['Q_L_atmos'] = ds.sel(smoothing='smooth_vel_tracer_atmos')
    decomp['Q_L_atmos_bar'] = filt(decomp['Q_L_atmos'])
    
    
    # Inferred Small scale
    decomp['Q_star'] = decomp['Q_H_bar'] - decomp['Q_L']
    decomp['Q_star_star'] = decomp['Q_H_bar'] - decomp['Q_L_bar']
    
    decomp['Q_star_ocean'] = decomp['Q_H_bar'] - decomp['Q_L_ocean']
    decomp['Q_star_ocean_bar'] = filt(decomp['Q_star_ocean'])
    decomp['Q_star_star_ocean'] = decomp['Q_H_bar'] - decomp['Q_L_ocean_bar']
    
    decomp['Q_star_atmos'] = decomp['Q_H_bar'] - decomp['Q_L_atmos']
    decomp['Q_star_atmos_bar'] = filt(decomp['Q_star_atmos'])
    decomp['Q_star_star_atmos'] = decomp['Q_H_bar'] - decomp['Q_L_atmos_bar']
    
    decomp['Q_star_res_wrong'] = decomp['Q_star'] - decomp['Q_star_star_ocean'] - decomp['Q_star_star_atmos']
    decomp['Q_star_res'] = decomp['Q_star'] - decomp['Q_star_ocean'] - decomp['Q_star_atmos']
    decomp['Q_star_star_res'] = decomp['Q_star_star'] - decomp['Q_star_star_ocean'] - decomp['Q_star_star_atmos'] 
    
    # for testing
    # decomp['Q_H_bar_bar'] = filt(decomp['Q_H_bar'])
    # decomp['Q_star_star_star'] = decomp['Q_H_bar_bar'] - decomp['Q_L_bar']
    # decomp['Q_star_res'] = decomp['Q_star'] - decomp['Q_star_ocean'] - decomp['Q_star_atmos']
    
    # concat into a single dataset
    datasets = [ds.drop([dvar for dvar in ['smoothing'] if dvar in ds]).assign_coords(term=k) for k,ds in decomp.items()]
    ds_out = xr.concat(datasets, dim='term', combine_attrs="override")
    ds_out.attrs = ds.attrs
    return ds_out

# an equivalent wrapper for coarsened data

In [6]:
ds_smooth_fluxes = open_zarr(mapper_smooth_fluxes)
ds_smooth_fluxes

<xarray.Dataset>
Dimensions:    (algo: 2, yt_ocean: 2700, xt_ocean: 3600, smoothing: 7, time: 300)
Coordinates: (12/13)
  * algo       (algo) <U5 'ncar' 'ecmwf'
    area_t     (yt_ocean, xt_ocean) float64 dask.array<chunksize=(2700, 3600), meta=np.ndarray>
    dxt        (yt_ocean, xt_ocean) float64 dask.array<chunksize=(2700, 3600), meta=np.ndarray>
    dyt        (yt_ocean, xt_ocean) float64 dask.array<chunksize=(2700, 3600), meta=np.ndarray>
    geolat_t   (yt_ocean, xt_ocean) float32 dask.array<chunksize=(2700, 3600), meta=np.ndarray>
    geolon_t   (yt_ocean, xt_ocean) float32 dask.array<chunksize=(2700, 3600), meta=np.ndarray>
    ...         ...
    kmt        (yt_ocean, xt_ocean) float32 dask.array<chunksize=(2700, 3600), meta=np.ndarray>
  * smoothing  (smoothing) <U23 'smooth_none' 'smooth_tracer' ... 'smooth_all'
  * time       (time) object 0181-01-01 12:00:00 ... 0181-10-27 12:00:00
    wet        (yt_ocean, xt_ocean) float64 dask.array<chunksize=(2700, 3600), meta=np.ndarray>
  * xt_ocean   (xt_ocean) float64 -279.9 -279.8 -279.7 ... 79.75 79.85 79.95
  * yt_ocean   (yt_ocean) float64 -81.11 -81.07 -81.02 ... 89.89 89.94 89.98
Data variables:
    evap       (algo, smoothing, time, yt_ocean, xt_ocean) float32 dask.array<chunksize=(1, 1, 3, 2700, 3600), meta=np.ndarray>
    qh         (algo, smoothing, time, yt_ocean, xt_ocean) float32 dask.array<chunksize=(1, 1, 3, 2700, 3600), meta=np.ndarray>
    ql         (algo, smoothing, time, yt_ocean, xt_ocean) float32 dask.array<chunksize=(1, 1, 3, 2700, 3600), meta=np.ndarray>
    taux       (algo, smoothing, time, yt_ocean, xt_ocean) float32 dask.array<chunksize=(1, 1, 3, 2700, 3600), meta=np.ndarray>
    tauy       (algo, smoothing, time, yt_ocean, xt_ocean) float32 dask.array<chunksize=(1, 1, 3, 2700, 3600), meta=np.ndarray>

In [7]:
smooth_decomp = decomposition_smooth(ds_smooth_fluxes)
smooth_decomp

<xarray.Dataset>
Dimensions:   (algo: 2, yt_ocean: 2700, xt_ocean: 3600, term: 20, time: 300)
Coordinates: (12/13)
  * algo      (algo) <U5 'ncar' 'ecmwf'
    area_t    (yt_ocean, xt_ocean) float64 dask.array<chunksize=(2700, 3600), meta=np.ndarray>
    dxt       (yt_ocean, xt_ocean) float64 dask.array<chunksize=(2700, 3600), meta=np.ndarray>
    dyt       (yt_ocean, xt_ocean) float64 dask.array<chunksize=(2700, 3600), meta=np.ndarray>
    geolat_t  (yt_ocean, xt_ocean) float32 dask.array<chunksize=(2700, 3600), meta=np.ndarray>
    geolon_t  (yt_ocean, xt_ocean) float32 dask.array<chunksize=(2700, 3600), meta=np.ndarray>
    ...        ...
    kmt       (yt_ocean, xt_ocean) float32 dask.array<chunksize=(2700, 3600), meta=np.ndarray>
  * time      (time) object 0181-01-01 12:00:00 ... 0181-10-27 12:00:00
    wet       (yt_ocean, xt_ocean) float64 dask.array<chunksize=(2700, 3600), meta=np.ndarray>
  * xt_ocean  (xt_ocean) float64 -279.9 -279.8 -279.7 ... 79.75 79.85 79.95
  * yt_ocean  (yt_ocean) float64 -81.11 -81.07 -81.02 ... 89.89 89.94 89.98
  * term      (term) <U17 'Q_H' 'Q_H_bar' ... 'Q_star_res' 'Q_star_star_res'
Data variables:
    evap      (term, algo, time, yt_ocean, xt_ocean) float32 dask.array<chunksize=(1, 1, 3, 2700, 3600), meta=np.ndarray>
    qh        (term, algo, time, yt_ocean, xt_ocean) float32 dask.array<chunksize=(1, 1, 3, 2700, 3600), meta=np.ndarray>
    ql        (term, algo, time, yt_ocean, xt_ocean) float32 dask.array<chunksize=(1, 1, 3, 2700, 3600), meta=np.ndarray>
    taux      (term, algo, time, yt_ocean, xt_ocean) float32 dask.array<chunksize=(1, 1, 3, 2700, 3600), meta=np.ndarray>
    tauy      (term, algo, time, yt_ocean, xt_ocean) float32 dask.array<chunksize=(1, 1, 3, 2700, 3600), meta=np.ndarray>

In [8]:
# import dask, distributed
# dask.config.set(
#     {
#         "distributed.worker.memory.target":False,
#         "distributed.worker.memory.spill":False
#         "logging.distributed"="debug",
#         "distributed.scheduler.worker-ttl": "1200s"
#     }
# )
# dask.config.get('distributed.worker.memory')

# # try a local cluster for testing
# from distributed import Client, LocalCluster
# cluster = LocalCluster(n_workers=4, threads_per_worker=4)
# client = Client(cluster)
# client

In [ ]:
from dask_gateway import Gateway
gateway = Gateway()

# close existing clusters
open_clusters = gateway.list_clusters()
print(list(open_clusters))
if len(open_clusters)>0:
    for c in open_clusters:
        cluster = gateway.connect(c.name)
        cluster.shutdown()
print('setting up new cluster')

options = gateway.cluster_options()
options.worker_memory = 52
options.worker_cores = 8

options.environment = dict(
    DASK_DISTRIBUTED__SCHEDULER__WORKER_SATURATION="1.0",
    # DASK_LOGGING__DISTRIBUTED="debug", # TODO: Testing this
    DASK_DISTRIBUTED__SCHEDULER__ALLOWED_FAILURES="5",
    DASK_DISTRIBUTED__SCHEDULER__IDLE_TIMEOUT="1200s",
    DASK_DISTRIBUTED__COMM__TIMEOUTS__TCP="1200s",
    DASK_DISTRIBUTED__COMM__TIMEOUTS__CONNECT="1200s",
    DASK_DISTRIBUTED__SCHEDULER__WORK_STEALING="False", # Test
    DASK_DISTRIBUTED__DEPLOY__LOST_WORKER_TIMEOUT="1200s",
    DASK_DISTRIBUTED__SCHEDULER__WORKER_TTL="1200s",
    DASK_DISTRIBUTED__WORKER__MEMORY__TARGET="false",
    DASK_DISTRIBUTED__WORKER__MEMORY__SPILL="false",
    DASK_DISTRIBUTED__WORKER__MEMORY__PAUSE=0.85,
)

# Create a cluster with those options
cluster = gateway.new_cluster(options)
client = cluster.get_client()

# cluster.adapt(10, 200)
cluster.scale(10)

def check_config():
    import dask
    return dask.config.get('distributed.worker.memory')

print(client.run(check_config))

client

[]
setting up new cluster


In [13]:
fs.rm(mapper_smooth_flux_decomposition.root, recursive=True)

In [ ]:
ds_save = smooth_decomp
print(f"{ds_save.nbytes/1e12}TB")
to_zarr_split(
    ds_save,
    mapper_smooth_flux_decomposition,
    split_interval=50
)

2.3332666142TB
Writing to leap-persistent/jbusecke/scale-aware-air-sea/results/CM26_fluxes_smoothed_decomposed_v0.6.2test.zarr ...


ERROR:tornado.application:Exception in callback <bound method BokehTornado._keep_alive of <bokeh.server.tornado.BokehTornado object at 0x7f8516c0dd20>>
Traceback (most recent call last):
  File "/srv/conda/envs/notebook/lib/python3.10/site-packages/tornado/ioloop.py", line 921, in _run
    val = self.callback()
  File "/srv/conda/envs/notebook/lib/python3.10/site-packages/bokeh/server/tornado.py", line 760, in _keep_alive
    c.send_ping()
  File "/srv/conda/envs/notebook/lib/python3.10/site-packages/bokeh/server/connection.py", line 93, in send_ping
    self._socket.ping(str(self._ping_count).encode("utf-8"))
  File "/srv/conda/envs/notebook/lib/python3.10/site-packages/tornado/websocket.py", line 444, in ping
    raise WebSocketClosedError()
tornado.websocket.WebSocketClosedError
2023-02-08 04:21:19,521 - tornado.application - ERROR - Exception in callback <bound method BokehTornado._keep_alive of <bokeh.server.tornado.BokehTornado object at 0x7f8516c0dd20>>
Traceback (most recent ca